## Music Generation Using CNN
dataset - midi(Music Dataset) https://www.kaggle.com/datasets/kritanjalijain/music-midi-dataset

In [6]:
import numpy as np
from music21 import converter, chord, instrument
import tensorflow as tf

In [11]:
import glob
midi_files= glob.glob("/Users/jellyfish/DeveloperLocal/DataScience2026/Music_CNN/archive/midi_dataset/**.mid")
print(len(midi_files))

50


In [16]:
from music21 import converter, instrument, chord
from music21 import note as m21_note  # Use an alias to avoid clashing with your variable

notes = []
for file in midi_files:
    try:
        midi = converter.parse(file)
        parts = instrument.partitionByInstrument(midi)
        
        if parts and len(parts.parts) > 0:
            notes_to_parse = parts.parts[0].recurse()
        else:
            notes_to_parse = midi.flat.notes
    
        for element in notes_to_parse:
            if isinstance(element, m21_note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder)) # Fixed the .apped typo

    except Exception as e:
        print(f"error reading {file}: {e}")
        
print("total notes: ", len(notes))
print(notes[:20])


total notes:  3892
['E6', 'E6', 'E-6', 'C#6', 'E6', 'E-6', 'E-6', 'C#6', 'B5', 'B5', 'C#6', 'C#6', 'B5', 'A5', 'B5', 'B5', 'A5', 'G#5', 'A5', 'A5']


In [17]:
pitchnames = sorted(set(notes))
print("total unique notes: ", len(pitchnames))

total unique notes:  113


In [18]:
notes_to_int = {note: number for number, note in enumerate(pitchnames)}
print(f"vocabulary size: {len(notes_to_int)}")

vocabulary size: 113


In [22]:
sequence_length = 100
network_input = []
network_output = []
for i in range(len(notes) - sequence_length):
    sequence_in = notes[i:i + sequence_length]
    sequence_out = notes[i + sequence_length]
    network_input.append([notes_to_int[n] for n in sequence_in])
    network_output.append(notes_to_int[sequence_out])

print("Input sequence : ", len(network_input))
print("Output sequence : ", len(network_output))




Input sequence :  3792
Output sequence :  3792


In [ ]:
n_pattern = len(network_input)
network_input = np.reshape(
    network_input,
    (n_pattern, sequence_length, 1)
)
network_input = network_input / float(len(pitchnames))
print("Final Reshaped Shape: ", network_input.shape)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3792,) + inhomogeneous part.